# Import

In [156]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [4]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [ ]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [133]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [134]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

## Helpful functions (big object, drop NAN)

In [8]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to HGNC

In [135]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi.get, c)) for c in comms]
    return comms_ncbi

In [136]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))

[['9679', '6671', '4430', '214', '30851', '23111', '64771', '3964', '54602', '687', '8682', '91966', '7205', '4212', '10614', '9204', '1200', '23151', '51741', '2274', '4072', '116068', '23180', '10868', '9414', '4134', '9711', '51136', '90355', '9910', '10807', '27289', '10788', '219333', '55317', '50862', '8835', '1050', '23034', '8204', '23621', '2055', '4059', '9590', '780', '10758', '22924', '115704', '8558', '9732', '138151', '9202', '9765', '58476', '128338', '23177', '81566', '961', '23204', '57222', '1808', '253725', '80381', '5570', '29780', '201161', '10150', '5997', '152137', '1396', '7016', '26268', '23612', '10169', '80762', '5218', '29946', '6738', '79016', '10123', '55810', '10950', '51330', '64112', '79956', '26036', '57798', '148170', '81671', '23271', '3308', '10391', '79845', '2887', '9098', '163590', '257364', '64089', '23623', '5801', '8609', '4681', '8572', '4929', '57381', '51255', '9891', '51119', '29923', '10106', '1363', '29775', '84952', '9520', '1620', '557

In [137]:
# NCBI to HGNC symbol
def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        mg = mygene.MyGeneInfo()
        entrez_ids = [str(e) for e in community]

        results = mg.querymany(
            entrez_ids,
            scopes="entrezgene",
            fields="symbol",
            species="human"
        )

        # Build a mapping: input ID -> symbol (or None)
        id_to_symbol = {}
        for r in results:
            q = str(r.get("query"))
            id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

        # Preserve original order
        symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
        comms_HGNC.append(symbols)
    return comms_HGNC


In [138]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

In [139]:
print(len(COMMUNITIES_HGNC))

16


In [140]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 1 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Community 15: dropped 0 NaN entries

Total dropped across all communities: 1


In [141]:
num_selected_comm = len(COMMUNITIES_HGNC)

In [142]:
print(num_selected_comm)

16


# Categoization Prep

### GO-slim

In [17]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [18]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [19]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [20]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [21]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [22]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [23]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=60)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [24]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

# Run Enrichment Analysis

### GO

In [143]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    category_counts_and_overlap_score_list = {}
    i = 0
    num_nonzero_communities = 0
    
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        

        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["GO_ID"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["GO_ID"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Compute overlap score for every category:
        filtered_exploded = filtered.explode('Category').reset_index(drop=True)
        category_counts_and_overlap_score = {}
        for val, group in filtered_exploded.groupby('Category'):
            overlap_list = group["Overlap"].tolist()
            numerators = [(lambda x: int(x.split("/")[0]))(e) for e in overlap_list]
            denominators = [(lambda x: int(x.split("/")[1]))(e) for e in overlap_list]
            overlap_score = sum(numerators)/sum(denominators)
            
            category_counts_and_overlap_score[val] = (len(group),overlap_score,)
        
        category_counts_and_overlap_score_list[i] = category_counts_and_overlap_score
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            print(category_counts_and_overlap_score)
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,category_counts_and_overlap_score_list

In [144]:
go_important_terms,go_category_counts_and_overlap_score = go_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE,slim_ids,depth = 1)

Size of community: 1001
Number of filtered terms: 45
Number of unmapped terms: 1
{'binding': (5, 0.16845329249617153), 'biological regulation': (9, 0.13252346769740475), 'catalytic activity': (7, 0.14783139890042762), 'cellular anatomical structure': (2, 0.13773584905660377), 'cellular process': (16, 0.15043731778425656), 'developmental process': (1, 0.10392609699769054), 'localization': (6, 0.15654648956356737), 'molecular function regulator activity': (2, 0.12440944881889764), 'protein-containing complex': (2, 0.15789473684210525)}


C:\Users\celem\AppData\Local\Temp\ipykernel_4760\1851936443.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
24,0,Negative Regulation Of Cilium Assembly (GO:1902018),7/14,0.000257,{GO:0065007},[biological regulation]
3871,0,Cul3-RING Ubiquitin Ligase Complex (GO:0031463),10/35,0.000252,{GO:0032991},[protein-containing complex]
3318,0,Phosphatidylinositol-3-Phosphate Binding (GO:0032266),11/40,0.000122,{GO:0005488},[binding]
16,0,Regulation Of TORC1 Signaling (GO:1903432),13/50,0.000127,{GO:0065007},[biological regulation]
9,0,Endocytic Recycling (GO:0032456),16/64,0.000022,"{GO:0051179, GO:0009987}","[localization, cellular process]"
22,0,Endosome Organization (GO:0007032),13/53,0.000214,{GO:0009987},[cellular process]
11,0,Vesicle-Mediated Transport To The Plasma Membrane (GO:0098876),18/83,0.000031,"{GO:0051179, GO:0009987}","[localization, cellular process]"
3310,0,Cysteine-Type Deubiquitinase Activity (GO:0004843),21/98,0.000001,{GO:0003824},[catalytic activity]
17,0,Regulation Of BMP Signaling Pathway (GO:0030510),16/75,0.000127,{GO:0065007},[biological regulation]
23,0,Protein Autoubiquitination (GO:0051865),15/71,0.000252,{GO:0009987},[cellular process]


Size of community: 1093
Number of filtered terms: 1
Number of unmapped terms: 0
{'transcription regulator activity': (1, 0.13168724279835392)}


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
1090,1,"DNA-binding Transcription Repressor Activity, RNA Polymerase II-specific (GO:0001227)",32/243,0.000333,{GO:0140110},[transcription regulator activity]


...

Size of community: 194
Number of filtered terms: 5
Number of unmapped terms: 0
{'molecular transducer activity': (1, 0.430939226519337), 'multicellular organismal process': (2, 0.4264705882352941), 'response to stimulus': (2, 0.425)}


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
8,11,Olfactory Receptor Activity (GO:0004984),156/362,3.659717e-249,{GO:0060089},[molecular transducer activity]
0,11,Sensory Perception Of Smell (GO:0007608),99/230,1.033201e-145,{GO:0032501},[multicellular organismal process]
1,11,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),60/141,5.703118e-85,{GO:0050896},[response to stimulus]
2,11,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),59/139,1.387560e-83,{GO:0050896},[response to stimulus]
3,11,Sensory Perception Of Chemical Stimulus (GO:0007606),46/110,2.196579e-64,{GO:0032501},[multicellular organismal process]


Size of community: 127
Number of filtered terms: 5
Number of unmapped terms: 0
{'cellular anatomical structure': (3, 0.2214765100671141), 'cellular process': (1, 0.36764705882352944), 'developmental process': (1, 0.1411764705882353)}


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
0,14,Intermediate Filament Organization (GO:0045109),25/68,5.460877e-37,{GO:0009987},[cellular process]
35,14,Keratin Filament (GO:0045095),12/39,1.366140e-16,{GO:0110165},[cellular anatomical structure]
37,14,Cornified Envelope (GO:0001533),9/41,1.982119e-11,{GO:0110165},[cellular anatomical structure]
36,14,Intermediate Filament (GO:0005882),12/69,1.312555e-13,{GO:0110165},[cellular anatomical structure]
2,14,Epidermis Development (GO:0008544),12/85,2.023896e-12,{GO:0032502},[developmental process]


9 out of 16 communities had significant GO terms.


In [145]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value)
0,0,1001,Negative Regulation Of Cilium Assembly (GO:190...,7/14,2.565064e-04,[biological regulation],GO_Biological_Process_2023,1.939704e-06,0.0,0.0,19.106640,251.309164,TCHP;LIMK2;TBC1D30;TESK1;CDK10;EVI5L;MAP4,GO:1902018,{GO:0065007},0.500000
1,0,1001,Cul3-RING Ubiquitin Ligase Complex (GO:0031463),10/35,2.523391e-04,[protein-containing complex],GO_Cellular_Component_2023,5.505581e-06,0.0,0.0,7.658527,92.742831,KLHL9;KLHL25;KCTD10;KLHL8;KCTD13;SPOPL;KLHL12;...,GO:0031463,{GO:0032991},0.285714
2,0,1001,Phosphatidylinositol-3-Phosphate Binding (GO:0...,11/40,1.219680e-04,[binding],GO_Molecular_Function_2023,2.831401e-06,0.0,0.0,7.268199,92.849349,SNX3;SNX4;ZFYVE26;SNX27;WDR45B;SH3PXD2B;SNX13;...,GO:0032266,{GO:0005488},0.275000
3,0,1001,Regulation Of TORC1 Signaling (GO:1903432),13/50,1.269753e-04,[biological regulation],GO_Biological_Process_2023,7.210448e-07,0.0,0.0,6.743243,95.366753,DYRK3;MIOS;PIH1D1;NPRL2;KICS2;ITFG2;RNF167;SES...,GO:1903432,{GO:0065007},0.260000
4,0,1001,Endocytic Recycling (GO:0032456),16/64,2.154898e-05,"[localization, cellular process]",GO_Biological_Process_2023,6.959862e-08,0.0,0.0,6.413198,105.692844,DENND1B;STX12;WASHC2C;VPS26A;PTPN23;SNX33;BLTP...,GO:0032456,"{GO:0051179, GO:0009987}",0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1310,14,127,Intermediate Filament Organization (GO:0045109),25/68,5.460877e-37,[cellular process],GO_Biological_Process_2023,2.022547e-38,0.0,0.0,113.030096,9810.320112,KRT82;KRT24;TCHH;KRT86;KRT85;KRT40;KRT84;KRT33...,GO:0045109,{GO:0009987},0.367647
1311,14,127,Keratin Filament (GO:0045095),12/39,1.366140e-16,[cellular anatomical structure],GO_Cellular_Component_2023,8.538377e-18,0.0,0.0,76.699517,3014.441398,KRT82;KRT36;KRT79;KRT78;KRT77;KRT76;KRT86;KRT7...,GO:0045095,{GO:0110165},0.307692
1312,14,127,Cornified Envelope (GO:0001533),9/41,1.982119e-11,[cellular anatomical structure],GO_Cellular_Component_2023,3.716473e-12,0.0,0.0,47.290519,1244.603513,SPRR2F;SPRR3;SPRR2G;TCHH;SPRR2A;DSG4;SPRR2B;DS...,GO:0001533,{GO:0110165},0.219512
1313,14,127,Intermediate Filament (GO:0005882),12/69,1.312555e-13,[cellular anatomical structure],GO_Cellular_Component_2023,1.640693e-14,0.0,0.0,36.276430,1151.452797,KRT82;KRT36;KRT79;KRT78;KRT77;KRT76;KRT75;KRT8...,GO:0005882,{GO:0110165},0.173913


### KEGG

In [146]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    category_counts_and_overlap_score_list = {}
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Compute overlap score for every category:
        filtered_exploded = filtered.explode('Category').reset_index(drop=True)
        category_counts_and_overlap_score = {}
        for val, group in filtered_exploded.groupby('Category'):
            overlap_list = group["Overlap"].tolist()
            numerators = [(lambda x: int(x.split("/")[0]))(e) for e in overlap_list]
            denominators = [(lambda x: int(x.split("/")[1]))(e) for e in overlap_list]
            overlap_score = sum(numerators)/sum(denominators)
            
            category_counts_and_overlap_score[val] = (len(group),overlap_score,)
        
        category_counts_and_overlap_score_list[i] = category_counts_and_overlap_score
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            print(category_counts_and_overlap_score)
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,category_counts_and_overlap_score_list

In [147]:
kegg_important_terms,kegg_category_counts_and_overlap_score = kegg_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1093
Number of filtered terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_4760\3460503351.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,1,Herpes simplex virus 1 infection,75/498,5.071605e-14,hsa05168,[Infectious disease: viral]


{'Infectious disease: viral': (1, 0.15060240963855423)}
Size of community: 1064
Number of filtered terms: 67


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
4,2,Adherens junction,27/71,4.077843e-15,hsa04520,[Cellular community - eukaryotes]
46,2,Bladder cancer,12/41,4.424133e-06,hsa05219,[Cancer: specific types]
24,2,p53 signaling pathway,21/73,1.016862e-09,hsa04115,[Cell growth and death]
15,2,Small cell lung cancer,26/92,1.533969e-11,hsa05222,[Cancer: specific types]
0,2,MAPK signaling pathway,80/294,9.196826e-33,hsa04010,[Signal transduction]
19,2,TGF-beta signaling pathway,25/94,1.467123e-10,hsa04350,[Signal transduction]
6,2,Ubiquitin mediated proteolysis,37/140,6.419128e-15,hsa04120,"[Folding, sorting and degradation]"
32,2,Melanoma,19/72,2.937696e-08,hsa05218,[Cancer: specific types]
18,2,AGE-RAGE signaling pathway in diabetic complications,26/100,1.058864e-10,hsa04933,[Endocrine and metabolic disease]
13,2,Leukocyte transendothelial migration,29/114,1.533969e-11,hsa04670,[Immune system]


...

{'Nervous system': (1, 0.12359550561797752), 'Signaling molecules and interaction': (1, 0.26392961876832843), 'Substance dependence': (2, 0.17557251908396945)}
Size of community: 194
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,11,Olfactory transduction,190/440,0.0,hsa04740,[Sensory system]


{'Sensory system': (1, 0.4318181818181818)}
Size of community: 127
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,14,Staphylococcus aureus infection,13/95,6.548994e-14,hsa05150,[Infectious disease: bacterial]


{'Infectious disease: bacterial': (1, 0.1368421052631579)}
6 out of 16 communities had significant GO terms.


In [148]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,KEGG_ID,Overlap (value)
0,1,1093,Herpes simplex virus 1 infection,75/498,5.071605e-14,[Infectious disease: viral],KEGG_2021_Human,1.035021e-15,0.0,0.0,3.219357,111.081819,ZNF177;ZNF133;ZNF254;ZNF573;ZNF253;ZNF175;ZNF2...,hsa05168,0.150602
1,2,1064,Adherens junction,27/71,4.077843e-15,[Cellular community - eukaryotes],KEGG_2021_Human,8.903588e-17,0.0,0.0,11.179188,413.154762,CTNND1;PTPRM;PTPRJ;IQGAP1;PTPRF;IGF1R;CDH1;RAC...,hsa04520,0.380282
2,2,1064,Bladder cancer,12/41,4.424133e-06,[Cancer: specific types],KEGG_2021_Human,9.080099e-07,0.0,0.0,7.436869,103.461800,RB1;RPS6KA5;CCND1;CDH1;CDKN2A;DAPK1;MMP1;CDK4;...,hsa05219,0.292683
3,2,1064,p53 signaling pathway,21/73,1.016862e-09,[Cell growth and death],KEGG_2021_Human,1.110112e-10,0.0,0.0,7.311822,167.597135,GADD45B;CDKN2A;GADD45A;IGFBP3;SIAH1;TNFRSF10B;...,hsa04115,0.287671
4,2,1064,Small cell lung cancer,26/92,1.533969e-11,[Cancer: specific types],KEGG_2021_Human,1.071769e-12,0.0,0.0,7.161499,197.383176,RB1;LAMA5;CDKN1B;MAX;LAMC1;CCND1;ITGAV;BAK1;CD...,hsa05222,0.282609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,9,516,Nicotine addiction,10/40,2.203426e-06,[Substance dependence],KEGG_2021_Human,5.065348e-08,0.0,0.0,12.815547,215.278861,GABRQ;GABRR2;GABRR1;GABRA6;GABRA5;GABRA4;GABRA...,hsa05033,0.250000
72,9,516,Morphine addiction,13/91,1.711104e-05,[Substance dependence],KEGG_2021_Human,5.900359e-07,0.0,0.0,6.430086,92.227256,GABRQ;KCNJ6;GABRA6;GABRA5;GABRA4;GABRA3;GABRG3...,hsa05032,0.142857
73,9,516,GABAergic synapse,11/89,3.915647e-04,[Nervous system],KEGG_2021_Human,1.800297e-05,0.0,0.0,5.419294,59.205647,GABRQ;GABRR2;GABRR1;KCNJ6;GABRA6;GABRA5;GABRA4...,hsa04727,0.123596
74,11,194,Olfactory transduction,190/440,0.000000e+00,[Sensory system],KEGG_2021_Human,0.000000e+00,0.0,0.0,3715.640000,inf,OR1C1;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;OR4K17;...,hsa04740,0.431818


### Reactome

In [151]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    category_counts_and_overlap_score_list= {}
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        reactome_level1 = build_reactome_level_map(level = 1)
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Compute overlap score for every category:
        filtered_exploded = filtered.explode('Category').reset_index(drop=True)
        category_counts_and_overlap_score = {}
        for val, group in filtered_exploded.groupby('Category'):
            overlap_list = group["Overlap"].tolist()
            numerators = [(lambda x: int(x.split("/")[0]))(e) for e in overlap_list]
            denominators = [(lambda x: int(x.split("/")[1]))(e) for e in overlap_list]
            overlap_score = sum(numerators)/sum(denominators)
            
            category_counts_and_overlap_score[val] = (len(group),overlap_score,)
        
        category_counts_and_overlap_score_list[i] = category_counts_and_overlap_score
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            print(category_counts_and_overlap_score)
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,category_counts_and_overlap_score_list

In [152]:
reactome_important_terms,reactome_category_counts_and_overlap_score = reactome_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 1001
Number of filtered terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_4760\2239647310.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,RHO GTPase Cycle R-HSA-9012999,53/441,0.000003,[Signal Transduction]


{'Signal Transduction': (1, 0.12018140589569161)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 1064
Number of filtered terms: 155


,Community Index,Term,Overlap,Adjusted P-value,Category
136,2,RUNX1 Regulates Expression Of Components Of Tight Junctions R-HSA-8935964,4/5,3.026653e-04,[Gene expression (Transcription)]
134,2,Drug-mediated Inhibition Of CDK4/CDK6 Activity R-HSA-9754119,4/5,3.026653e-04,[Cell Cycle]
135,2,PTK6 Expression R-HSA-8849473,4/5,3.026653e-04,[Signal Transduction]
154,2,Fibronectin Matrix Formation R-HSA-1566977,4/6,7.587939e-04,[Extracellular matrix organization]
155,2,PTK6 Regulates Cell Cycle R-HSA-8849470,4/6,7.587939e-04,[Signal Transduction]
156,2,PTK6 Promotes HIF1A Stabilization R-HSA-8857538,4/6,7.587939e-04,[Signal Transduction]
121,2,Cross-presentation Of Particulate Exogenous Antigens (Phagosomes) R-HSA-1236973,5/8,1.840713e-04,[Immune System]
107,2,Endosomal/Vacuolar Pathway R-HSA-1236977,6/11,8.250573e-05,[Immune System]
150,2,RUNX3 Regulates p14-ARF R-HSA-8951936,5/10,6.117315e-04,[Gene expression (Transcription)]
81,2,Aberrant Regulation Of Mitotic G1/S Transition In Cancer Due To RB1 Defects R-HSA-9659787,8/17,1.293666e-05,[Disease]


...

{'Disease': (3, 0.16574585635359115), 'Neuronal System': (1, 0.1694915254237288), 'Signal Transduction': (11, 0.17994858611825193)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 194
Number of filtered terms: 3


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,182/393,5.935693e-314,[Sensory Perception]
1,11,Olfactory Signaling Pathway R-HSA-381753,182/401,3.972307e-312,[Sensory Perception]
2,11,Sensory Perception R-HSA-9709957,182/616,5.559541e-270,[Sensory Perception]


{'Sensory Perception': (3, 0.3872340425531915)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 127
Number of filtered terms: 3


,Community Index,Term,Overlap,Adjusted P-value,Category
0,14,Keratinization R-HSA-6805567,118/208,1.219960e-238,[Developmental Biology]
2,14,Formation Of Cornified Envelope R-HSA-6809371,23/74,9.002650e-33,[Developmental Biology]
1,14,Developmental Biology R-HSA-1266738,118/1073,1.418930e-139,[Developmental Biology]


{'Developmental Biology': (3, 0.19114391143911438)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
7 out of 16 communities had significant GO terms.


In [153]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1001,RHO GTPase Cycle R-HSA-9012999,53/441,2.705489e-06,[Signal Transduction],Reactome_2022,3.213170e-09,0.0,0.0,2.681671,5.244278e+01,DOCK5;ANKLE2;DOCK4;ABCD3;RASGRF2;ITSN1;FAF2;AR...,0.120181
1,2,1064,RUNX1 Regulates Expression Of Components Of Ti...,4/5,3.026653e-04,[Gene expression (Transcription)],Reactome_2022,3.814641e-05,0.0,0.0,71.452830,7.269667e+02,TJP1;CLDN5;OCLN;CBFB,0.800000
2,2,1064,Drug-mediated Inhibition Of CDK4/CDK6 Activity...,4/5,3.026653e-04,[Cell Cycle],Reactome_2022,3.814641e-05,0.0,0.0,71.452830,7.269667e+02,CCND2;CDK6;CCND1;CDK4,0.800000
3,2,1064,PTK6 Expression R-HSA-8849473,4/5,3.026653e-04,[Signal Transduction],Reactome_2022,3.814641e-05,0.0,0.0,71.452830,7.269667e+02,EPAS1;PTK6;NR3C1;HIF1A,0.800000
4,2,1064,Fibronectin Matrix Formation R-HSA-1566977,4/6,7.587939e-04,[Extracellular matrix organization],Reactome_2022,1.095958e-04,0.0,0.0,35.724528,3.257617e+02,CEACAM1;CEACAM6;FN1;ITGA5,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,11,194,Olfactory Signaling Pathway R-HSA-381753,182/401,3.972307e-312,[Sensory Perception],Reactome_2022,2.648205e-312,0.0,0.0,1356.481735,9.731843e+05,OR1C1;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;OR4K17;...,0.453865
189,11,194,Sensory Perception R-HSA-9709957,182/616,5.559541e-270,[Sensory Perception],Reactome_2022,5.559541e-270,0.0,0.0,676.978495,4.197148e+05,OR1C1;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;OR4K17;...,0.295455
190,14,127,Keratinization R-HSA-6805567,118/208,1.219960e-238,[Developmental Biology],Reactome_2022,1.016633e-239,0.0,0.0,2881.967901,1.585951e+06,KRTAP24-1;LCE1A;LIPM;KRTAP3-3;KRTAP3-2;KRTAP3-...,0.567308
191,14,127,Formation Of Cornified Envelope R-HSA-6809371,23/74,9.002650e-33,[Developmental Biology],Reactome_2022,2.250662e-33,0.0,0.0,85.955128,6.461598e+03,SPRR2F;SPRR3;SPRR2G;SPINK6;TCHH;KLK13;KLK14;LC...,0.310811


### Disease Data Sets

In [ ]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [ ]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms df

In [228]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
0,0,1001,Negative Regulation Of Cilium Assembly (GO:190...,7/14,2.565064e-04,[biological regulation],GO_Biological_Process_2023,1.939704e-06,0.0,0.0,19.106640,2.513092e+02,TCHP;LIMK2;TBC1D30;TESK1;CDK10;EVI5L;MAP4,GO:1902018,{GO:0065007},0.500000,NaN
25,0,1001,Early Endosome (GO:0005769),43/307,9.160869e-08,[cellular anatomical structure],GO_Cellular_Component_2023,9.993675e-10,0.0,0.0,3.185317,6.601220e+01,WASHC4;MGRN1;TGFBRAP1;SNX13;WASHC2C;VPS26A;SNX...,GO:0005769,{GO:0110165},0.140065,NaN
26,0,1001,Ubiquitin-Protein Transferase Activity (GO:000...,57/412,1.712309e-09,[catalytic activity],GO_Molecular_Function_2023,3.057695e-12,0.0,0.0,3.171127,8.407722e+01,RNF10;TRAF3IP2;PPP1R11;UBE3C;RNF19B;UBE2Z;TRIM...,GO:0004842,{GO:0003824},0.138350,NaN
27,0,1001,GTPase Activator Activity (GO:0005096),29/211,3.979882e-05,[molecular function regulator activity],GO_Molecular_Function_2023,7.817625e-07,0.0,0.0,3.084684,4.337595e+01,RABGAP1;RGS14;NPRL2;AGAP2;ARHGAP1;HACD3;IQGAP2...,GO:0005096,{GO:0098772},0.137441,NaN
28,0,1001,Lytic Vacuole (GO:0000323),30/223,5.531503e-05,[cellular anatomical structure],GO_Cellular_Component_2023,8.045822e-07,0.0,0.0,3.010517,4.224642e+01,WDR45B;VPS26A;TMEM97;CD1D;ZFYVE26;SNX2;DRAM1;L...,GO:0000323,{GO:0110165},0.134529,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313,14,127,Intermediate Filament (GO:0005882),12/69,1.312555e-13,[cellular anatomical structure],GO_Cellular_Component_2023,1.640693e-14,0.0,0.0,36.276430,1.151453e+03,KRT82;KRT36;KRT79;KRT78;KRT77;KRT76;KRT75;KRT8...,GO:0005882,{GO:0110165},0.173913,NaN
1314,14,127,Epidermis Development (GO:0008544),12/85,2.023896e-12,[developmental process],GO_Biological_Process_2023,2.248773e-13,0.0,0.0,28.302561,8.242617e+02,LCE2B;CASP14;SPRR2F;SPRR3;SPRR2G;KLK14;KRT32;T...,GO:0008544,{GO:0032502},0.141176,NaN
1582,14,127,Formation Of Cornified Envelope R-HSA-6809371,23/74,9.002650e-33,[Developmental Biology],Reactome_2022,2.250662e-33,0.0,0.0,85.955128,6.461598e+03,SPRR2F;SPRR3;SPRR2G;SPINK6;TCHH;KLK13;KLK14;LC...,NaN,NaN,0.310811,NaN
1581,14,127,Keratinization R-HSA-6805567,118/208,1.219960e-238,[Developmental Biology],Reactome_2022,1.016633e-239,0.0,0.0,2881.967901,1.585951e+06,KRTAP24-1;LCE1A;LIPM;KRTAP3-3;KRTAP3-2;KRTAP3-...,NaN,NaN,0.567308,NaN


In [246]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [ ]:
def run_enrichment_func(community,term_score_cap,percentage):
    # GO df
    enr_go = gp.enrichr(
        gene_list=community,
        gene_sets=['GO_Biological_Process_2023',
                'GO_Molecular_Function_2023',
                'GO_Cellular_Component_2023'],
        organism='Human',
        outdir=None # don't write to disk
    )
    GO_df = enr_go.results
    mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
    GO_df = GO_df[mask].copy()   
    
    # KEGG df
    enr_kegg = gp.enrichr(
        gene_list=community,
        gene_sets=['KEGG_2021_Human'],
        organism='Human',
        outdir=None
    )
    KEGG_df = enr_kegg.results
    mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
    KEGG_df = KEGG_df[mask].copy() 
       
    # Reactome df
    enr_reactome = gp.enrichr(
        gene_list=community,
        gene_sets=['Reactome_2022'],
        organism='Human',
        outdir=None
    )
    Reactome_df = enr_reactome.results  
    mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
    Reactome_df = Reactome_df[mask].copy()
    
    
    all_df = [GO_df,KEGG_df,Reactome_df]
    # build result df by concatenating
    result = pd.concat(all_df, ignore_index=True)
    return result

In [ ]:
from json import JSONDecodeError

# ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
_ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
    """
    Calls user's run_enrichment_func(community) with retries + memoization.
    Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
    """
    # Ensure we always pass a list of gene symbols (never a bare string)
    genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
    if len(genes) == 0:
        return pd.DataFrame()

    key = tuple(sorted(genes))
    if key in _ENR_CACHE:
        return _ENR_CACHE[key].copy()

    for a in range(retries):
        try:
            df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
            if df is None:
                # treat as transient failure to trigger retry
                raise RuntimeError("run_enrichment_func returned None")
            _ENR_CACHE[key] = df.copy()
            return df
        except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
            # Transient errors from HTTP/JSON/file handling inside gseapy
            if a == retries - 1:
                # Give up: return empty so pipeline continues
                return pd.DataFrame()
            time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

    return pd.DataFrame()

# ---------------- 2) Minimal bootstrap to record robust terms ----------------
def get_robust_terms(communities_HGNC, run_enrichment_func,
                     R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
    """
    Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
    Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
    """
    rng = np.random.default_rng(seed)
    rows = []

    for cid, community in enumerate(communities_HGNC):
        n = len(community)
        if n == 0:
            continue
        drop_k = max(1, int(np.floor(leaveout * n)))
        counts = Counter()

        for _ in range(R):
            # Jackknife subset (ensure not empty)
            keep = np.ones(n, dtype=bool)
            keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
            sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
            if len(sub) == 0:
                continue

            df = run_enrichment_safe(run_enrichment_func, sub)
            if df is None or df.empty:
                continue

            # Your function already returns significant terms; just count them.
            # If it includes multiple libraries, preserve Gene_set to disambiguate names.
            if 'Term' not in df.columns:
                continue  # be defensive

            if 'Gene_set' in df.columns:
                terms = (df[['Term', 'Gene_set']]
                         .dropna()
                         .drop_duplicates()
                         .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
                         .tolist())
            else:
                terms = df['Term'].dropna().drop_duplicates().tolist()

            counts.update(terms)

            # tiny pause helps with API rate limits if your func calls Enrichr internally
            time.sleep(0.03)

        # Keep only robust terms
        for t, c in counts.items():
            freq = c / max(R, 1)
            if freq >= recurrence_cutoff:
                if '|' in t:
                    term, gene_set = t.split('|', 1)
                    rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
                else:
                    rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

    return (pd.DataFrame(rows)
              .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
              .reset_index(drop=True))

In [ ]:
twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
                                R=25, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
twr3

In [ ]:
terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
                                R=10, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
terms_with_recurrence

In [ ]:
# rename important terms to match terms_with_recurrence
important_terms = important_terms.rename(columns={'index': 'community_id'})
important_terms = important_terms.rename(columns={'Term': 'term'})

In [ ]:
terms_with_rec_merged = important_terms.merge(
    terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
    on=['community_id', 'term', 'Gene_set'],
    how='left'
)

terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

terms_with_rec_merged = terms_with_rec_merged.sort_values(
    ['community_id', 'recurrence'],
    ascending=[True, False]
).reset_index(drop=True)

In [ ]:
terms_with_rec_merged

In [ ]:
community_summary = (
    terms_with_rec_merged
    .groupby("community_id")["recurrence"]
    .agg(mean_recurrence="mean", term_count="count")
    .reset_index()
)

print(community_summary)

In [ ]:
display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [ ]:
for c in communities:
    print(len(c))

In [ ]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [ ]:
all_comms_ncbi = index_to_ncbi(communities,index_to_gene_distinct)

In [ ]:
print(all_comms_ncbi)

In [ ]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))

In [ ]:
for c in all_comms_ncbi:
    print(len(c),DGIDB_count(c))

In [ ]:
def tbd(id):
    print(len(communities[id]))
    c8_ncbi = index_to_ncbi([communities[id]])[0]
    print(len(c8_ncbi))
    print(DGIDB_count(c8_ncbi))

In [ ]:
def tbd_selected(id):
    print(len(communities_selected[id]))
    c8_ncbi = index_to_ncbi([communities_selected[id]])[0]
    print(len(c8_ncbi))
    print(DGIDB_count(c8_ncbi))

In [ ]:
tbd_selected(1)